# 04 - Calidad de los datos

Acá medimos qué falta y qué está mal en cada tabla (sobre bronce) y qué proponemos hacer con eso (numeral 5). Arrestos, choques y vehículos se revisan solo dentro del periodo de estudio; pobreza, SAT y PUMA se revisan completos.

Cuenta como faltante un nulo, un texto vacío o el texto `(null)`. Además buscamos faltantes "disfrazados", es decir, celdas que tienen algo pero no dicen nada: coordenadas (0, 0), `UNKNOWN`, `Unspecified` o `s`.

Algunas de las soluciones ya se aplican en el 05 y las demás quedan para la entrega final.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pyspark.sql import functions as F

import geo
from codigos import (ARRESTOS_BORO, ARRESTOS_GRAVEDAD, ARRESTOS_GRUPOS_EDAD,
                     POBREZA_COLUMNAS_DICCIONARIO, SAT_BORO_DBN)
from config import BRONZE, FECHA_INICIO, FECHA_FIN, sesion
from estilo import CATEGORICA, TINTA_SECUNDARIA, guardar

spark = sesion("04_calidad")

VACIOS = ["", "(null)"]

def faltantes(df, tabla):
    """nulos por columna en una sola pasada"""
    exprs = [F.sum(F.when(F.col(c).isNull() | F.trim(F.col(c)).isin(VACIOS), 1).otherwise(0)).alias(c) for c in df.columns]
    fila = df.agg(F.count("*").alias("__n"), *exprs).first().asDict()
    n = fila.pop("__n")
    out = pd.DataFrame({"tabla": tabla, "columna": list(fila), "faltantes": list(fila.values())})
    out["pct"] = (100 * out["faltantes"] / n).round(2)
    out.attrs["filas"] = n
    return out

def conteo(df, columna, top=15):
    """frecuencia de cada valor"""
    return (df.groupBy(columna).count().orderBy(F.desc("count")).limit(top).toPandas()
              .assign(pct=lambda d: (100 * d["count"] / d["count"].sum()).round(2)))

resumen = []

## 1. Arrestos (histórico + 2026)

In [ ]:
arr = (spark.read.parquet(f"{BRONZE}/arrestos_historico").drop("lon_lat")
       .unionByName(spark.read.parquet(f"{BRONZE}/arrestos_2026").drop("location")))
n_total = arr.count()
arr = arr.withColumn("_fecha", F.to_date("arrest_date", "MM/dd/yyyy"))
fechas_invalidas = arr.filter(F.col("_fecha").isNull()).count()
arr = arr.filter(F.col("_fecha").between(FECHA_INICIO, FECHA_FIN)).drop("_fecha").cache()
f_arr = faltantes(arr, "arrestos")
print(f"Registros totales (2006-2026): {n_total:,} | en el periodo: {f_arr.attrs['filas']:,} | fechas no convertibles: {fechas_invalidas:,}")
f_arr[f_arr["faltantes"] > 0].sort_values("pct", ascending=False)

### Faltantes disfrazados y valores inválidos en arrestos

In [ ]:
lat, lon = F.col("latitude").cast("double"), F.col("longitude").cast("double")
chequeos_arr = arr.agg(
    F.sum(((lat == 0) | (lon == 0)).cast("int")).alias("coordenada_en_cero"),
    F.sum((lat.isNotNull() & (lat != 0) & ~geo.coordenada_valida(lat, lon)).cast("int")).alias("coordenada_fuera_de_NYC"),
    F.sum((~F.col("age_group").isin(ARRESTOS_GRUPOS_EDAD) | F.col("age_group").isNull()).cast("int")).alias("age_group_invalido"),
    F.sum((~F.col("law_cat_cd").isin(list(ARRESTOS_GRAVEDAD)) | F.col("law_cat_cd").isNull()).cast("int")).alias("law_cat_cd_no_documentado"),
    F.sum((~F.col("arrest_boro").isin(list(ARRESTOS_BORO)) | F.col("arrest_boro").isNull()).cast("int")).alias("arrest_boro_invalido"),
    (F.count("*") - F.countDistinct("arrest_key")).alias("arrest_key_duplicados"),
).toPandas().T.rename(columns={0: "registros"})
chequeos_arr["pct"] = (100 * chequeos_arr["registros"] / f_arr.attrs["filas"]).round(3)
chequeos_arr

In [ ]:
print("Valores de AGE_GROUP (los que no están en", ARRESTOS_GRUPOS_EDAD, "son inválidos):")
display(conteo(arr, "age_group", 20))
print("Valores de LAW_CAT_CD (documentados: F, M, V):")
display(conteo(arr, "law_cat_cd"))
print("Valores de PERP_RACE (UNKNOWN es un faltante disfrazado):")
display(conteo(arr, "perp_race"))

¿En qué años aparece `AGE_GROUP = (null)`? Si se concentra en pocos años, lo más probable es que haya cambiado la forma de registrarlo.

In [ ]:
(arr.withColumn("anio", F.substring("arrest_date", 7, 4))
    .groupBy("anio").agg(F.count("*").alias("arrestos"),
                         F.sum((F.col("age_group") == "(null)").cast("int")).alias("age_group_null"))
    .withColumn("pct", F.round(100 * F.col("age_group_null") / F.col("arrestos"), 2))
    .orderBy("anio").toPandas())

## 2. Choques

In [ ]:
ch = spark.read.parquet(f"{BRONZE}/choques").withColumn("_fecha", F.to_date("crash_date", "MM/dd/yyyy"))
n_total_ch = ch.count()
ch = ch.filter(F.col("_fecha").between(FECHA_INICIO, FECHA_FIN)).drop("_fecha").cache()
f_ch = faltantes(ch, "choques")
print(f"Registros totales (2012-2026): {n_total_ch:,} | en el periodo: {f_ch.attrs['filas']:,}")
f_ch[f_ch["faltantes"] > 0].sort_values("pct", ascending=False)

In [ ]:
lat, lon = F.col("latitude").cast("double"), F.col("longitude").cast("double")
sin_borough = (F.col("borough").isNull() | (F.trim("borough") == ""))
chequeos_ch = ch.agg(
    F.sum(sin_borough.cast("int")).alias("sin_borough"),
    F.sum((sin_borough & geo.coordenada_valida(lat, lon)).cast("int")).alias("sin_borough_pero_con_coordenadas_validas"),
    F.sum(((lat == 0) | (lon == 0)).cast("int")).alias("coordenada_en_cero"),
    F.sum(((lon < -180) | (lon > 180) | (lat < -90) | (lat > 90)).cast("int")).alias("coordenada_imposible"),
    F.sum((lat.isNotNull() & (lat != 0) & ~geo.coordenada_valida(lat, lon)).cast("int")).alias("coordenada_fuera_de_NYC"),
    F.sum((F.col("contributing_factor_vehicle_1") == "Unspecified").cast("int")).alias("factor_1_Unspecified"),
    F.sum(F.col("number_of_persons_injured").isNull().cast("int")).alias("heridos_vacio"),
    (F.count("*") - F.countDistinct("collision_id")).alias("collision_id_duplicados"),
).toPandas().T.rename(columns={0: "registros"})
chequeos_ch["pct"] = (100 * chequeos_ch["registros"] / f_ch.attrs["filas"]).round(3)
chequeos_ch

- `sin_borough_pero_con_coordenadas_validas`: a estos se les puede recuperar el borough con el spatial join del notebook 05.
- `factor_1_Unspecified`: la celda tiene un valor, pero no dice la causa, así que lo contamos como faltante.

In [ ]:
# grafico de faltantes
g = f_ch[f_ch["pct"] > 0].sort_values("pct")
fig, ax = plt.subplots(figsize=(8, 0.28 * len(g) + 1))
ax.barh(g["columna"], g["pct"], color=CATEGORICA[0], height=0.7)
for y, v in zip(g["columna"], g["pct"]):
    ax.text(v + 0.5, y, f"{v:.1f} %", va="center", fontsize=7, color=TINTA_SECUNDARIA)
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.set_xlim(0, 105)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=7)
ax.set_title("Choques: porcentaje de valores faltantes por columna (2019 – jun 2026)")
guardar(fig, "04_faltantes_choques")
plt.show()

## 3. Vehículos

In [ ]:
ve = spark.read.parquet(f"{BRONZE}/vehiculos").withColumn("_fecha", F.to_date("crash_date", "MM/dd/yyyy"))
ve = ve.filter(F.col("_fecha").between(FECHA_INICIO, FECHA_FIN)).drop("_fecha").cache()
f_ve = faltantes(ve, "vehiculos")
print(f"Registros en el periodo: {f_ve.attrs['filas']:,}")
f_ve[f_ve["faltantes"] > 0].sort_values("pct", ascending=False)

In [ ]:
anio_v = F.col("vehicle_year").cast("int")
ids_choques = ch.select(F.col("collision_id").alias("_cid")).distinct()
huerfanos = ve.join(ids_choques, ve["collision_id"] == ids_choques["_cid"], "left_anti").count()
chequeos_ve = ve.agg(
    F.sum((anio_v.isNotNull() & ~anio_v.between(1900, 2027)).cast("int")).alias("anio_vehiculo_imposible"),
    F.sum(F.col("vehicle_id").rlike("^[0-9]+$").cast("int")).alias("vehicle_id_numerico"),
    F.sum(F.col("vehicle_id").rlike("^[0-9a-f]{8}-").cast("int")).alias("vehicle_id_formato_uuid"),
    (F.count("*") - F.countDistinct("unique_id")).alias("unique_id_duplicados"),
).toPandas().T.rename(columns={0: "registros"})
chequeos_ve.loc["sin_choque_en_la_tabla_de_choques"] = huerfanos
chequeos_ve["pct"] = (100 * chequeos_ve["registros"] / f_ve.attrs["filas"]).round(3)
chequeos_ve

`VEHICLE_ID` no sirve como llave porque cambió de formato con los años. Por eso usamos `UNIQUE_ID`, y la unión con choques se hace por `COLLISION_ID`.

## 4. Pobreza

In [ ]:
po = spark.read.parquet(f"{BRONZE}/pobreza")
f_po = faltantes(po, "pobreza")
print(f"Personas en la muestra: {f_po.attrs['filas']:,}")
display(f_po[f_po["faltantes"] > 0].sort_values("pct", ascending=False))

# columnas del archivo vs diccionario
en_archivo = set(po.columns)
dicc = {c.lower(): c for c in POBREZA_COLUMNAS_DICCIONARIO}
print("En el archivo y NO en el diccionario:", sorted(en_archivo - set(dicc)))
print("En el diccionario y NO en el archivo:", sorted(dicc[c] for c in set(dicc) - en_archivo))

In [ ]:
# ¿el nivel educativo falta solo en menores? (AgeCateg 1 = menor de 18)
(po.groupBy("agecateg")
   .agg(F.count("*").alias("personas"),
        F.sum(F.when(F.col("educattain").isNull() | (F.trim("educattain") == ""), 1).otherwise(0)).alias("sin_educacion"))
   .orderBy("agecateg").toPandas())

In [ ]:
# faltantes estructurales: si los vacíos son solo de menores, es "no aplica"
edad = F.col("agep").cast("int")
filas = []
for c in ["eng", "jwtr", "wkw", "esr", "msp", "lanx", "schl", "educattain"]:
    falta = F.col(c).isNull() | (F.trim(F.col(c)) == "")
    r = po.agg(F.sum(falta.cast("int")).alias("faltantes"),
               F.min(F.when(falta, edad)).alias("edad_min_sin_dato"),
               F.max(F.when(falta, edad)).alias("edad_max_sin_dato"),
               F.min(F.when(~falta, edad)).alias("edad_min_con_dato")).first().asDict()
    filas.append({"columna": c, **r})
pd.DataFrame(filas)

Si `edad_max_sin_dato` es menor que `edad_min_con_dato`, el faltante es estructural: la pregunta no aplica para esas edades. Si los rangos se cruzan, el faltante tiene otra causa.

In [ ]:
# códigos fuera de rango
po.agg(
    F.sum((~F.col("boro").isin("1", "2", "3", "4", "5")).cast("int")).alias("boro_fuera_de_1_a_5"),
    F.sum((~F.col("nycgov_pov_stat").isin("1", "2")).cast("int")).alias("pov_stat_fuera_de_1_2"),
    F.sum((F.col("pwgtp").cast("int") <= 0).cast("int")).alias("peso_no_positivo"),
).toPandas()

## 5. SAT 2012

In [ ]:
sat = spark.read.parquet(f"{BRONZE}/sat_2012")
f_sat = faltantes(sat, "sat_2012")
puntajes = ["sat_critical_reading_avg_score", "sat_math_avg_score", "sat_writing_avg_score"]
sat.agg(
    F.count("*").alias("colegios"),
    *[F.sum((F.col(c) == "s").cast("int")).alias(f"{c}_s") for c in puntajes],
    F.sum((~F.col("dbn").rlike("^[0-9]{2}[MXKQR][0-9]{3}$")).cast("int")).alias("dbn_formato_inesperado"),
    F.sum((~F.substring("dbn", 3, 1).isin(list(SAT_BORO_DBN))).cast("int")).alias("letra_borough_invalida"),
).toPandas().T.rename(columns={0: "valor"})

## 6. Resumen por tabla

In [ ]:
tablas = [f_arr, f_ch, f_ve, f_po, f_sat]
resumen = pd.DataFrame([{
    "tabla": t["tabla"].iloc[0],
    "filas_evaluadas": t.attrs["filas"],
    "columnas": len(t),
    "columnas_con_faltantes": int((t["faltantes"] > 0).sum()),
    "celdas_faltantes_pct": round(100 * t["faltantes"].sum() / (t.attrs["filas"] * len(t)), 2),
    "columna_mas_incompleta": t.sort_values("pct").iloc[-1]["columna"] if t["pct"].max() > 0 else "—",
    "pct_mas_incompleta": t["pct"].max(),
} for t in tablas])
pd.concat(tablas).to_csv("../figuras/tabla_faltantes_por_columna.csv", index=False)
resumen.to_csv("../figuras/tabla_calidad_resumen.csv", index=False)
resumen

## 7. Técnicas propuestas

| Problema | Tabla | Qué hacemos | Cuándo |
| --- | --- | --- | --- |
| Borough vacío | Choques | Recuperarlo con el spatial join de lat/lon contra las PUMAs 2010. Si no hay coordenadas, queda como "sin dato" | Ya en el 05 |
| Coordenadas (0, 0), imposibles o fuera de NYC | Arrestos, choques | Pasan a nulo, pero el registro se deja para los análisis que no son espaciales | Ya en el 05 |
| `AGE_GROUP` fuera de las categorías (como `895`) | Arrestos | Pasa a nulo ("Desconocido") | Ya en el 05 |
| `LAW_CAT_CD` no documentado (`I`, `9`, vacío) | Arrestos | Se agrupa como "Otro / no documentado" y no entra en la comparación felonía vs. delito menor | Ya en el 05 |
| `(null)` escrito como texto (en `AGE_GROUP`, `LAW_CAT_CD`) | Arrestos | Se trata como nulo | Ya en el 05 |
| `AGE_GROUP` sin dato concentrado en un solo año (ver la tabla por año) | Arrestos | Sacar ese año de los análisis por edad o reportarlo aparte | Entrega final |
| `UNKNOWN` en raza o sexo | Arrestos | Dejarlo como categoría "Desconocido", sin imputar | Entrega final |
| Factor contribuyente `Unspecified` o vacío | Choques, vehículos | Contarlo como faltante al analizar causas y reportar qué % es | Entrega final |
| Heridos o muertos vacíos | Choques | Poner 0 solo si los demás conteos del choque están; si no, sacarlo del modelo | Entrega final |
| Vehículos sin choque asociado | Vehículos | Dejarlos por fuera cuando se cruza con choques | Ya en el 05 (columna `tiene_choque`) |
| Año del vehículo imposible | Vehículos | Pasa a nulo | Ya en el 05 |
| Puntajes `s` (suprimidos) | SAT | No se imputan: se sacan del promedio del borough y se reporta cuántos colegios quedaron por fuera | Ya en el 05 |
| Nivel educativo vacío en menores | Pobreza | No es un error: la educación se calcula solo con adultos | Entrega final |
| Columnas del diccionario que no están en el archivo / `EST_EITC` sin documentar | Pobreza | Usar solo las columnas documentadas y mencionar la inconsistencia | Documento |

## Conclusiones

- Toda la calidad se midió en el cluster, con una sola pasada por tabla.
- El problema más grande es el borough que falta en los choques, que se puede recuperar con las coordenadas cuando las hay.
- Hay faltantes disfrazados que no se ven si solo se cuentan nulos.
- No inventamos datos: lo inválido pasa a nulo o a una categoría explícita.